# Pelajaran 01 - Pengenalan Agen AI

Selamat datang di pelajaran pertama dalam kursus **Agen AI untuk Pemula**!

**Agen AI** adalah program yang menggunakan model bahasa besar (LLM) sebagai mesin penalarannya dan dapat mengambil *tindakan* di dunia nyata — memanggil API, melakukan query database, atau menjalankan kode — untuk mencapai tujuan atas nama pengguna.

Dalam notebook ini Anda akan membangun agen pertama Anda: sebuah **Agen Perjalanan** yang merekomendasikan tujuan liburan. Sepanjang jalan Anda akan belajar bagaimana:

1. Terhubung ke Layanan Agen Azure AI Foundry menggunakan **Microsoft Agent Framework**.
2. Memberi agen sebuah **alat** — fungsi Python biasa yang dapat dipanggilnya.
3. Menjalankan agen dan memeriksa responsnya.
4. Menyalurkan respons agen token demi token.


## Setup

Sebelum menjalankan notebook ini, pastikan Anda telah:

1. **Sebuah proyek Azure AI Foundry** dengan model chat yang sudah dideploy (misalnya `gpt-4o-mini`).
2. **Masuk dengan Azure CLI** — jalankan `az login` di terminal Anda.
3. **Mengatur variabel lingkungan yang diperlukan:**
   - `AZURE_AI_PROJECT_ENDPOINT` — endpoint proyek Azure AI Foundry Anda.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — nama model yang sudah Anda deploy.

Sel berikut menginstal paket Python yang Anda butuhkan.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Membuat Agen Pertama Anda

Sebuah agen membutuhkan dua hal:

- **Instruksi** yang memberitahunya *siapa dirinya* dan *bagaimana cara berperilaku* (prompt sistem).
- **Alat** — fungsi Python yang dihias dengan `@tool` yang dapat dipanggil oleh agen untuk mengambil informasi atau melakukan tindakan.

Di bawah ini kami mendefinisikan alat sederhana yang mengembalikan daftar tujuan liburan populer. Agen akan menggunakan alat ini ketika pengguna meminta rekomendasi perjalanan.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Untuk pengalaman yang lebih interaktif, Anda dapat **streaming** respons agen. Alih-alih menunggu balasan penuh, agen memberikan potongan teks saat mereka dihasilkan. Ini sangat berguna dalam antarmuka obrolan di mana Anda ingin menampilkan output secara real time.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Ringkasan

Dalam pelajaran ini Anda belajar bagaimana:

- **Membuat penyedia** yang terhubung ke Azure AI Foundry Agent Service melalui `AzureAIProjectAgentProvider`.
- **Mendefinisikan sebuah alat** menggunakan dekorator `@tool` sehingga agen dapat memanggil fungsi Python Anda.
- **Menjalankan agen** dengan pesan pengguna dan mencetak responsnya.
- **Mengalirkan respons** untuk keluaran waktu nyata.

Dalam pelajaran berikutnya kita akan menjelajahi kerangka kerja agen lebih mendalam dan belajar bagaimana memberi agen alat yang lebih kuat serta kemampuan penalaran multi-langkah.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berusaha untuk akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber otoritatif. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
